# Python + SQL I

This notebook demonstrates how to use SQL from Python using the `sqlite3` module. You will learn how to connect to a database, run queries, read results, and write data - all from Python code.

For more details, see the [sqlite3 documentation](https://docs.python.org/3/library/sqlite3.html).

## Learning Objectives

By the end of this lecture, you should be able to:

- Connect Python to a SQLite database and close the connection when finished.
- Execute parameterized SQL queries and process returned tuples using Python.
- Explain why reading results exhausts a cursor.
- Create a table, insert rows, and commit changes.
- Explain what a SQLite connection context manager does and what it does not do.

## Setup

Run the cells in order. Locally, open this notebook from its lecture directory so `data/warehouse.db` points to the bundled database. In Colab, the setup downloads the database when needed. `Path` describes the file's location; we will study file operations in a later lecture.

The warehouse database contains warehouses, products, carriers, and shipments linking them. The reading examples use this database; the writing examples create a separate, disposable `data/tutorial.db`.

In [ ]:
import sqlite3
from pathlib import Path
from urllib.request import urlretrieve

# Download the warehouse database if not present (e.g., on Colab)
db_path = Path("data/warehouse.db")
if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/olearydj/INSY3010/main/lectures/12b-python-sql-i/data/warehouse.db"
    urlretrieve(url, db_path)
    print("Downloaded warehouse.db")
else:
    print(f"Using {db_path}")

## Connecting to a Database

The `sqlite3` module is included with Python - no installation needed. Working with a database requires two objects: a *connection* and a *cursor*.

In [ ]:
import sqlite3

# create a connection to the database file
con = sqlite3.connect(db_path)

# create a cursor - this is what we use to run queries
cur = con.cursor()

The connection represents the link to the database file. The cursor is the object we use to execute queries and retrieve results.

## Reading Data

### Running a Query

Use `cur.execute()` to run a SQL statement, then retrieve results with `fetchall()`.

In [ ]:
cur.execute("SELECT * FROM Warehouse")
rows = cur.fetchall()

# results are a list of tuples
print(type(rows))
print(type(rows[0]))

In [ ]:
# each tuple is one row from the database
for row in rows:
    print(row)

### Cursor Exhaustion

A cursor is an *iterator* - once you've read through the results, they're gone. Fetching or looping consumes the current query's results. Execute the query again if you need a fresh result set.

In [ ]:
cur.execute("SELECT * FROM Carrier")

# loop through results
for row in cur:
    print(row)

In [ ]:
# now try fetchall - the cursor is already exhausted
remaining = cur.fetchall()
print(remaining)  # empty list!

If you want to reuse all the rows, call `fetchall()` after `execute()` and store the results in a variable. Looping directly over the cursor is also useful when you only need one pass. Then you can use that variable as many times as you need.

In [ ]:
cur.execute("SELECT * FROM Carrier")
carriers = cur.fetchall()  # store results right away

# now use the list freely
print(f"Found {len(carriers)} carriers")
print(f"First: {carriers[0]}")
print(f"Last: {carriers[-1]}")

### Multi-Line Queries

Use triple-quoted strings for longer queries. This keeps your SQL readable.

In [ ]:
SQL = """
SELECT w.name, COUNT(*) AS shipment_count
FROM Warehouse w
JOIN Shipment s ON w.warehouse_id = s.warehouse_id
GROUP BY w.warehouse_id, w.name
ORDER BY shipment_count DESC
"""

cur.execute(SQL)
results = cur.fetchall()

for row in results:
    print(row)

### Column Names

Query results are tuples - they don't carry column names. Use `cur.description` to get the column names from the most recent query.

In [ ]:
# cur.description holds metadata about the last query
for col_info in cur.description:
    print(col_info[0])  # first element is the column name

### Formatting Output

Combining column names with data makes output more readable.

In [ ]:
# re-run to get fresh results
cur.execute("""
SELECT w.name, w.city, w.state, w.capacity_sqft
FROM Warehouse w
ORDER BY w.capacity_sqft DESC
""")
results = cur.fetchall()

# get column names
columns = []
for col in cur.description:
    columns.append(col[0])
print(columns)

print()
for row in results:
    print(f"  {row[0]} ({row[1]}, {row[2]}) - {row[3]:,} sq ft")

### Parameterized Queries

When a query depends on a variable - like user input or a function argument - use `?` placeholders instead of f-strings. This keeps supplied values separate from SQL instructions and avoids SQL injection through those values. Placeholders stand for values, not table or column names.

In [ ]:
# BAD - never do this (vulnerable to SQL injection)
# category = "Electronics"
# cur.execute(f"SELECT * FROM Product WHERE category = '{category}'")

# GOOD - use ? placeholders
category = "Electronics"
cur.execute("SELECT product_name, unit_weight, unit_cost FROM Product WHERE category = ?",
            (category,))
results = cur.fetchall()

print(f"Products in '{category}':")
for name, weight, cost in results:
    print(f"  {name} - {weight} lbs, ${cost:.2f}")

With `?` placeholders, pass a sequence of values, such as a tuple or list, as the second argument to `execute()`. We use tuples here. The comma in `(category,)` makes a one-item tuple; `(category)` alone is just the original value. Supply one value for each placeholder.

In [ ]:
# multiple parameters
min_weight = 5.0
max_cost = 50.0

cur.execute("""
    SELECT product_name, category, unit_weight, unit_cost
    FROM Product
    WHERE unit_weight > ? AND unit_cost < ?
    ORDER BY unit_weight DESC
""", (min_weight, max_cost))

for name, cat, weight, cost in cur.fetchall():
    print(f"  {name} ({cat}) - {weight} lbs, ${cost:.2f}")

### Exercise 1 - Products in a Category

Using the open warehouse connection, select `product_name` and `unit_cost` for a category stored in a Python variable. Use a `?` parameter and order by `product_name`. Fetch the rows, print each name and cost, and use a Python loop to compute the sum of the returned unit costs. This sum is not inventory value; no quantities are involved.

In [ ]:
category = "Electronics"
# Write your query and process its results here.

#### Solution / Discussion

In [ ]:
cur.execute("""
    SELECT product_name, unit_cost FROM Product
    WHERE category = ? ORDER BY product_name
""", (category,))
products = cur.fetchall()
total_cost = 0
for name, cost in products:
    print(f"{name}: ${cost:.2f}")
    total_cost += cost
print(f"Sum of unit costs: ${total_cost:.2f}")

What would change if you selected a category with no matching products? The result would be an empty list and the total would remain zero.

### Close the Connection

Always close the connection when you're done.

In [ ]:
con.close()

## Writing Data

So far we've only read from an existing database. You can also create tables and insert data from Python.

### Creating a Table

We'll create a small database from scratch. First, connect to a new file - `sqlite3.connect()` creates the file if it doesn't exist.

In [ ]:
import os

# start fresh each time this section runs
if os.path.exists("data/tutorial.db"):
    os.remove("data/tutorial.db")

con = sqlite3.connect("data/tutorial.db")
cur = con.cursor()

# create a table
cur.execute("CREATE TABLE movie (title TEXT, year INTEGER, score REAL)")

# insert a couple of rows
SQL = """
INSERT INTO movie VALUES
    ('Monty Python and the Holy Grail', 1975, 8.2),
    ('And Now for Something Completely Different', 1972, 7.5)
"""
cur.execute(SQL)

# IMPORTANT: commit saves the changes to the database file
con.commit()

With the connection settings used here, inserts start a transaction: a group of pending changes. `con.commit()` saves those changes. Closing the connection with uncommitted inserts discards them. Reading through the same connection can show pending changes, so that alone does not prove they were saved.

In [ ]:
# inspect the rows after committing
cur.execute("SELECT * FROM movie")
for row in cur.fetchall():
    print(row)

### Inserting Multiple Rows

Use `executemany()` with `?` placeholders to insert multiple rows from a list. This is both safer (parameterized) and cleaner than building SQL strings by hand.

In [ ]:
more_movies = [
    ("Monty Python Live at the Hollywood Bowl", 1982, 7.9),
    ("Monty Python's The Meaning of Life", 1983, 7.5),
    ("Monty Python's Life of Brian", 1979, 8.0),
]

cur.executemany("INSERT INTO movie VALUES (?, ?, ?)", more_movies)
con.commit()

# verify
cur.execute("SELECT year, title, score FROM movie ORDER BY year")
for year, title, score in cur.fetchall():
    print(f"  {year} - {title} ({score})")

### Exercise 2 - Add and Verify a Movie

Insert `("The Example Movie", 2026, 7.2)` into `movie` using placeholders, commit, and query that title to verify its values. Run your insertion once; rerunning an insert adds another row because this table has no uniqueness constraint. Rerun the Writing Data section from Creating a Table to reset the disposable database.

In [ ]:
# Insert, commit, and query the new movie here.

#### Solution / Discussion

Run either your insertion or this solution on a fresh table to avoid adding the same example twice.

In [ ]:
new_movie = ("The Example Movie", 2026, 7.2)
cur.execute("INSERT INTO movie VALUES (?, ?, ?)", new_movie)
con.commit()
cur.execute("SELECT title, year, score FROM movie WHERE title = ?", (new_movie[0],))
print(cur.fetchall())

In [ ]:
con.close()

## Context Managers - Automatic Commit or Rollback

A SQLite connection context manager handles the **commit, not the close**. Instead of calling `con.commit()` yourself, put the database changes inside `with con:`. When the block finishes successfully, pending changes are committed automatically. If an error escapes the block, the pending transaction is rolled back instead; the error still propagates.

A context manager supplies behavior around an indented `with` block, and that behavior depends on the object. **Keep calling `con.close()` when you are done with the database.** This differs from file handling: as we will soon see, `with open(...) as file:` closes the file when its block ends. For SQLite connections, `with con:` manages the transaction and leaves the connection open.

In [ ]:
# Explicit commit (the pattern used above)
con = sqlite3.connect("data/tutorial.db")
cur = con.cursor()
cur.execute("INSERT INTO movie VALUES (?, ?, ?)", ("Spamalot", 2005, 7.0))
con.commit()
con.close()

In [ ]:
# with con: replaces the explicit commit, not the close.
con = sqlite3.connect("data/tutorial.db")
with con:
    cur = con.cursor()
    cur.execute("INSERT INTO movie VALUES (?, ?, ?)", ("A New Example", 2026, 6.8))

# The connection is still open, so we can verify the result.
cur.execute("SELECT title, year FROM movie ORDER BY year DESC, title LIMIT 3")
for title, year in cur.fetchall():
    print(f"  {title} ({year})")
con.close()

If an insertion fails inside `with con:`, the pending transaction is rolled back. An unhandled error also stops the cell, so the later `con.close()` line is not guaranteed to run. Handling that error and guaranteeing closure are topics for Files & Exceptions. On the successful path shown here, we close explicitly after using the results.

## Summary

| Task | Method |
|------|--------|
| Connect to database | `sqlite3.connect(path)` |
| Create a cursor | `con.cursor()` |
| Run a query | `cur.execute(sql)` |
| Get all results | `cur.fetchall()` |
| Get one result | `cur.fetchone()` |
| Column names | `cur.description` |
| Parameterized query | `cur.execute(sql, (val1, val2))` |
| Insert many rows | `cur.executemany(sql, data_list)` |
| Save changes | `con.commit()` |
| Commit or roll back a transaction | `with con:` |
| Close connection | `con.close()` |


## Common Gotchas

- `fetchall()` returns a list of tuples, even if each row has just one column. An empty result is `[]`.
- A cursor's results are consumed as you read them. Store rows if you need to reuse them.
- Match parameter values to `?` placeholders and remember the comma in a one-item tuple.
- `commit()` saves pending changes; `close()` releases the connection. Neither is a substitute for the other.
- `with con:` manages transactions, not connection closure.
- Rerunning an insert can create duplicate rows. Reset the tutorial database before repeating the full writing demonstration.

## Glossary

| Term | Meaning |
|---|---|
| Connection | Python object representing access to a database |
| Cursor | Object used to execute SQL and retrieve results |
| Parameter | A value supplied separately from the SQL text |
| Transaction | A group of database changes that can be committed or rolled back together |
| Commit | Save a transaction's changes |
| Rollback | Undo a transaction's pending changes |
| Context manager | An object that supplies entry/exit behavior for a `with` block |

## Project Preview - Last Semester

Last semester's project used this warehouse database. Students compared SQL processing with Python processing, implemented functions, exported results to CSV, and explained their work in a written reflection. Examples included warehouse shipment totals and delayed shipments by carrier. Today's lecture introduces the database access those tasks required; file export and error handling come later. This is a historical preview, not the Fall project specification.

## Practice Problems

Complete these independently after the examples. Use the existing `db_path` for the warehouse database. Open and close your own connections for each problem; do not change the warehouse data. The code cells are blank so running the notebook does not require completed answers.

### ★ 1. Warehouse Names

Write `warehouse_names(db_path, state)` to return a list of warehouse names in the supplied state, sorted alphabetically. Use a parameterized query and build the list from the returned tuples. The table is `Warehouse`, with columns `name` and `state`. Return `[]` if there are no matches. Check a state shown in the earlier examples and a nonexistent state such as `"ZZ"`.

In [ ]:
# Define and try warehouse_names here.

### ★★ 2. Affordable Products

Write `affordable_products(db_path, category, max_cost)` to return a list of `(product_name, unit_cost)` tuples for products in the supplied category whose unit cost is at most `max_cost`, ordered by cost and then name. Use two query parameters. Check a normal category and budget, a zero budget, and a category with no matches. Use Python to print a numbered report from the returned list.

In [ ]:
# Define and try affordable_products here.

### ★★ 3. Save and Reopen

Create a separate database at `data/practice.db` containing a table `reading (label TEXT, value REAL)`. Begin with a fresh table each time: execute `DROP TABLE IF EXISTS reading` before `CREATE TABLE`. This reset deletes the previous practice readings.

Insert `[("A", 12.5), ("B", 8.0), ("C", 15.0)]` with `executemany()` inside `with con:`. Close the connection, reopen the database, and query readings whose value is at least a parameterized threshold of `10`. Order by label; the expected result is `[("A", 12.5), ("C", 15.0)]`. Close the reopened connection. Explain why reopening is a stronger persistence check than querying through the original connection.

In [ ]:
# Create, insert, close, reopen, and verify here.

---

Auburn University / Industrial and Systems Engineering  
INSY 3010 / Programming and Databases for ISE / Fall 2026  
© Copyright 2026, Danny J. O'Leary.  
For licensing, attribution, and information: [GitHub INSY3010](https://github.com/olearydj/INSY3010)